# 00 · Data coverage (Phase 0)

Can we get **at least 5 years of daily Tarbela and Mangla inflow with under 10% of days missing**? Everything else in the project depends on the answer.

Run these first, from the repo root:

```
python -m pipeline.probe_pmd
python -m pipeline.fetch_weather
```

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from pipeline import coverage_report as cr

df = cr.load_coverage()
y = cr.yearly(df)
v = cr.verdict(y)
print(v.summary)

## Coverage per year

In [ ]:
y.style.format({c: "{:.0%}" for c in ["tarbela_missing", "mangla_missing", "worst_missing"]})

## Missing-days heatmap

In [ ]:
from IPython.display import Image
Image(filename=str(cr.plot_heatmap(df)))

## Months that failed to download or parse cleanly

In [ ]:
df[(~df.status.isin(["ok", "cached"])) | (df.problems.fillna("") != "")][["month", "status", "tarbela_days", "mangla_days", "problems"]]

## Weather CSVs

In [ ]:
from pipeline.config import WEATHER_DIR
rows = []
for p in sorted(WEATHER_DIR.glob("era5_*.csv")):
    w = pd.read_csv(p, parse_dates=["date"])
    rows.append({"point": p.stem[5:], "first": w.date.min().date(), "last": w.date.max().date(),
                 "days": len(w), "missing_tmean": int(w.tmean_c.isna().sum())})
pd.DataFrame(rows)

## Decision

*Write the go / no-go call here once the numbers are in, and what you'll do about any gaps (see Risks in the project plan).*